# 02 — Data Cleaning and Validation

**Goal:** show what `src/data/validation.py` and `src/data/cleaning.py`
actually catch, using a deliberately corrupted copy of the data so every
check has something to find.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 5)

np.random.seed(42)

from notebooks.nb_utils import fetch_price_panel

TICKERS = ["AAPL", "MSFT", "GLD", "BND", "SPY"]
START = "2019-01-01"

price_panel, used_synthetic = fetch_price_panel(TICKERS, START)
returns_panel = price_panel.pct_change().dropna()
print(f"Loaded {price_panel.shape[0]} rows x {price_panel.shape[1]} tickers. Synthetic data: {used_synthetic}")
price_panel.tail()

Failed to get ticker 'MSFT' reason: Expecting value: line 1 column 1 (char 0)


HTTP Error 403: Host not in allowlist: query2.finance.yahoo.com. Add this host to your network egress settings to allow access.


HTTP Error 403: Host not in allowlist: query1.finance.yahoo.com. Add this host to your network egress settings to allow access.


Failed to get ticker 'GLD' reason: Expecting value: line 1 column 1 (char 0)


HTTP Error 403: Host not in allowlist: query2.finance.yahoo.com. Add this host to your network egress settings to allow access.


HTTP Error 403: Host not in allowlist: query1.finance.yahoo.com. Add this host to your network egress settings to allow access.


... (7 outputs left out)

['MSFT', 'GLD']: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')


['BND', 'AAPL', 'SPY']: possibly delisted; no timezone found


[info] Live data unavailable (KeyError); using synthetic demo data instead.
Loaded 2004 rows x 5 tickers. Synthetic data: True


,AAPL,MSFT,GLD,BND,SPY
2026-08-31,229.681016,128.274867,119.914845,364.746718,107.728124
2026-09-01,227.093540,128.294744,120.000708,372.445526,106.711406
2026-09-02,229.155329,127.875189,121.085078,365.765614,106.991434
2026-09-03,232.958620,125.652809,122.697759,374.610608,107.656847
2026-09-04,232.705684,125.622649,120.900524,369.667262,106.661264


## Inject realistic data problems

We corrupt a copy of the raw panel with: a duplicate row, a negative price, an impossible OHLC relationship, and a big (but real-looking) single-day return — the last one should be **flagged, not deleted**, since real crashes are legitimate data.

In [2]:
long_df = price_panel.reset_index().melt(id_vars="index", var_name="ticker", value_name="adj_close")
long_df = long_df.rename(columns={"index": "date"})
long_df["open"] = long_df["adj_close"]
long_df["high"] = long_df["adj_close"] * 1.01
long_df["low"] = long_df["adj_close"] * 0.99
long_df["close"] = long_df["adj_close"]
long_df["volume"] = 1_000_000

corrupted = long_df.copy()
# 1) duplicate a row
corrupted = pd.concat([corrupted, corrupted.iloc[[10]]], ignore_index=True)
# 2) negative price
corrupted.loc[50, "close"] = -5.0
# 3) impossible OHLC (low > high)
corrupted.loc[100, "low"] = corrupted.loc[100, "high"] + 10
# 4) a big, but legitimate-looking, one-day drop
corrupted.loc[200, "adj_close"] = corrupted.loc[199, "adj_close"] * 0.65

print(f"Original rows: {len(long_df)}, corrupted rows: {len(corrupted)}")

Original rows: 10020, corrupted rows: 10021


In [3]:
from src.data.validation import validate_prices

report = validate_prices(corrupted)
print(report.summary())

duplicate_asset_date: 2 row(s)
negative_or_zero_prices: 1 row(s)
impossible_ohlc: 2 row(s)
suspicious_returns: 2 row(s)
non_monotonic_dates: 2005 row(s)


Note the big one-day drop shows up under `suspicious_returns` — flagged for review, not silently removed. Duplicates, negative prices, and impossible OHLC rows are caught too.

In [4]:
from src.data.cleaning import clean_prices

cleaned = clean_prices(corrupted)
print(f"Corrupted: {len(corrupted)} rows -> Cleaned: {len(cleaned)} rows")

# Confirm the "crash" day survived cleaning (legitimate large move, not an error)
crash_day = long_df.loc[200, "date"]
survived = cleaned[(cleaned["date"] == crash_day)]
print(f"\nCrash-day row survived cleaning: {not survived.empty}")

Corrupted: 10021 rows -> Cleaned: 10018 rows

Crash-day row survived cleaning: True


## Takeaways

- Data errors (duplicates, negative prices, impossible OHLC) are removed.
- Large but real market moves are preserved — cleaning never uses a return-size threshold to decide what's an error.
- See `docs/DATA_LEAKAGE_AUDIT.md` and the module docstring in `cleaning.py` for the full list of documented decisions.
- Continue to **03 — Exploratory Data Analysis**.